# 01 - Orientation and Mental Model

Before performing complex topological analyses, develop a mental model of pySurgery's core concepts:
1. **Three-level representation**: Geometric (SimplicialComplex) → Algebraic (CWComplex) → Purely algebraic (ChainComplex)
2. **Result objects**: Every computation returns `{status, exact, validated, missing_data, ...}`
3. **Exactness-first**: We use ℤ arithmetic by default; approximation is opt-in and risky
4. **Caching**: Intermediate results are cached to avoid redundant computation
5. **Contracts**: All code aligns with `CONTRACT_VERSION` for reproducibility

## Learning Goals

- Build small topological spaces from scratch
- Compute Betti numbers, Euler characteristic, homology groups exactly
- Read and interpret result objects confidently
- Understand when exact vs approximate computation is appropriate
- Verify alignment with pySurgery's contract version

## Setup

In [1]:
import pysurgery as ps
from pysurgery.bridge.julia_bridge import julia_engine

if julia_engine.available:
    julia_engine.warmup()

print('='*70)
print('pySurgery 01: Orientation and Mental Model')
print(f'Version: {getattr(ps, "__version__", "unknown")}')
print(f'CONTRACT: {ps.CONTRACT_VERSION}')
print('='*70 + '\n')

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
pySurgery 01: Orientation and Mental Model
Version: 1.2.1
CONTRACT: 2026.04-phase10



## Part 1: Three-Level Representation

pySurgery encodes topological spaces at three abstraction levels:
1. **Geometric**: SimplicialComplex (vertices, edges, faces, etc.)
2. **Combinatorial**: CWComplex (cells with attaching maps)
3. **Algebraic**: ChainComplex (boundary operators as matrices)

### Toy Example: Single Triangle

In [6]:
print('TOY: Single triangle')
triangle = ps.SimplicialComplex.from_maximal_simplices([(0, 1, 2)])
print(f'  f-vector: {triangle.f_vector()}')
print(f'  Euler χ: {triangle.euler_characteristic()}')

cc = triangle.cellular_chain_complex()
cc.homology()
print(f'  H_0 rank: {cc.betti_number(0)}, H_1 rank: {cc.betti_number(1)}, H_2 rank: {cc.betti_number(2)}')

/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:884: UserWarning: Torsion certification may be incomplete for this complex; the sparse integer reduction returned only unit factors, so torsion could not be fully resolved.
  warnings.warn(


TOY: Single triangle
  f-vector: {0: 3, 1: 3, 2: 1}
  Euler χ: 1
  H_0 rank: 1, H_1 rank: 0, H_2 rank: 0


### Medium Example: Circle

In [7]:
print('\nMEDIUM: Circle (S^1) via edge loop')
circle = ps.SimplicialComplex.from_maximal_simplices([(0,1), (1,2), (2,0)])
print(f'  f-vector: {circle.f_vector()}')
print(f'  Euler χ: {circle.euler_characteristic()}')
cc_circle = circle.cellular_chain_complex()
print('  H_0(S^1) = Z (connected)')
print('  H_1(S^1) = Z (one independent loop)')


MEDIUM: Circle (S^1) via edge loop
  f-vector: {0: 3, 1: 3}
  Euler χ: 0
  H_0(S^1) = Z (connected)
  H_1(S^1) = Z (one independent loop)


## Summary

✓ Understand the three levels of representation
✓ Can build simplicial complexes from scratch
✓ Compute exact invariants (Betti, Euler)

**Next**: Continue to `02_building_spaces_simplicial_and_cw.ipynb`

## --- Integration from Original: 01_basic_homology_and_cohomology.ipynb ---

This section contains the deep-dive visualization, plotting functions, and didactic examples merged from the original `01_basic_homology_and_cohomology.ipynb`.

## 1. From Discrete Data to Simplicial Complexes

In the real world, you rarely are handed a perfect mathematical equation of a manifold. Instead, you are handed **discrete data**: a point cloud from a 3D scanner, a dataset of high-dimensional vectors, or a network of nodes.

How do we do topology on discrete points? A point cloud has no topology; it's just disconnected dots (zero-dimensional topology). To find the shape of the data, we must connect the dots. We do this by building a **Simplicial Complex**.

- **Vietoris-Rips Complex**: Imagine growing a ball of radius $\epsilon$ around every point in your dataset. If two balls overlap, you draw an edge (a 1-simplex) between the points. If three balls mutually overlap, you fill in the triangle between them (a 2-simplex). If $k$ balls overlap, you fill in a $(k-1)$-dimensional solid simplex. This transforms discrete points into a continuous geometric shape!
- **Alpha Complex**: Similar to Rips, but geometrically constrained by the Delaunay triangulation. It is much more memory efficient for 2D and 3D data because it prevents the combinatorial explosion of high-dimensional simplices.

While `pysurgery` interfaces with libraries like `gudhi` to build these complexes from data (as we will see in Tutorial 4), at its core, `pysurgery` operates on the purely algebraic skeleton of these shapes: **The Chain Complex**.

## 2. The Chain Complex and Boundary Matrices

Once we have a Simplicial Complex (or a more general CW Complex), we organize its building blocks by dimension:
- $C_0$: The set of all vertices (0-cells).
- $C_1$: The set of all edges (1-cells).
- $C_2$: The set of all triangles/faces (2-cells).
- $C_n$: The set of all $n$-dimensional solid bodies.

To understand the topology, we must know exactly how an $n$-dimensional cell is glued to the $(n-1)$-dimensional cells. We represent this mathematically using the **Boundary Operator**, denoted as $\partial_n$ or $d_n$.

$$ d_n : C_n \to C_{n-1} $$

### How do we build the Boundary Matrix?
Suppose you have a triangle (a 2-simplex) made of vertices $[v_0, v_1, v_2]$. Its boundary consists of three edges: $[v_0, v_1]$, $[v_1, v_2]$, and $[v_0, v_2]$.

**Orientation is the most critical concept here.** If we just say the boundary is edge 1 + edge 2 + edge 3, we lose track of "direction." If you walk around the perimeter of the triangle, you traverse $[v_0, v_1]$, then $[v_1, v_2]$, but you traverse $[v_0, v_2]$ *backwards*. Therefore, the algebraic boundary is:
$$ \partial([v_0, v_1, v_2]) = [v_1, v_2] - [v_0, v_2] + [v_0, v_1] $$

Notice the alternating signs! $+1, -1, +1$. This alternating sum guarantees a fundamental theorem of topology: **The boundary of a boundary is zero** ($d_{n-1} \circ d_n = 0$).

In code, $d_n$ is a sparse matrix. The rows represent the $(n-1)$-cells, and the columns represent the $n$-cells. The entries are $0, 1,$ or $-1$ depending on how they are glued.

In [8]:
import numpy as np
from scipy.sparse import csr_matrix
from pysurgery.core.complexes import CWComplex

print("Libraries loaded. Ready to build a manifold.")

Libraries loaded. Ready to build a manifold.


## 3. Manually Constructing The Real Projective Plane ($\mathbb{RP}^2$)

To see how `pysurgery` handles complex algebra without needing a million data points, we will manually define a famous topological space: **The Real Projective Plane ($\mathbb{RP}^2$)**.

$\mathbb{RP}^2$ is a non-orientable surface. You cannot embed it in 3D space without it intersecting itself (like a closed Möbius strip).

We can build it using exactly 3 cells:
- One 0-cell: A vertex $v$.
- One 1-cell: An edge $e$.
- One 2-cell: A solid disk $f$.

**The Gluing Instructions (The Math):**
1. We take the edge $e$ and glue both of its ends to the single vertex $v$. This makes $e$ a loop. Thus, its boundary is $v - v = 0$. So, $d_1 = [0]$.
2. We take the solid disk $f$. Its boundary is a circle. We wrap this boundary circle around the loop $e$ **twice**. Thus, the boundary of $f$ is $2e$. So, $d_2 = [2]$.

Let's write this in code.

In [9]:
# Define the number of cells in each dimension
cells = {0: 1, 1: 1, 2: 1}

# Construct the Boundary Matrices using Scipy Sparse Matrices.
# We use sparse matrices because in real TDA data, these matrices can have millions of rows/cols,
# but are 99.9% zeros. Dense matrices would instantly crash your RAM.
d1 = csr_matrix((1, 1), dtype=np.int64)  # Matrix of zeros, shape (1,1)
d2 = csr_matrix([[2]], dtype=np.int64)  # Matrix with a single '2', shape (1,1)

attaching_maps = {1: d1, 2: d2}

# Instantiate the CWComplex
rp2 = CWComplex(cells=cells, attaching_maps=attaching_maps)

# Extract the purely algebraic Chain Complex
chain = rp2.cellular_chain_complex()
print("Chain Complex for RP^2 created successfully.")

Chain Complex for RP^2 created successfully.


## 4. The Algebra of Homology: Cycles vs Boundaries

Homology $H_n(X)$ is the mathematical formalization of "counting holes".

- **Cycles ($Z_n$)**: A collection of $n$-cells that have no boundary. For example, a closed loop of edges has no start or end point. Mathematically, these are the vectors in the **kernel** (nullspace) of $d_n$. ($Z_n = \ker(d_n)$).
- **Boundaries ($B_n$)**: A collection of $n$-cells that form the boundary of an $(n+1)$-cell. For example, the outer shell of a solid sphere is a cycle, but it is also a boundary (it's filled in). These are the vectors in the **image** (column space) of $d_{n+1}$. ($B_n = \operatorname{im}(d_{n+1})$).

A "hole" exists if there is a cycle that is **not** a boundary. You have a loop, but there is no solid disk filling it in.

Therefore, the Homology group is the quotient space:
$$ H_n(X) = Z_n / B_n = \ker(d_n) / \operatorname{im}(d_{n+1}) $$

### Why is computing this so hard? (The Smith Normal Form)
If we were doing linear algebra over the Real numbers $\mathbb{R}$, finding dimensions of kernels and images is easy (just use Gaussian Elimination / RREF). 

But we are doing topology. We must work over the **Integers $\mathbb{Z}$**. The integers form a Principal Ideal Domain (PID), not a Field. You cannot divide by 2. If you have $2x = 0$, over $\mathbb{R}$, $x=0$. Over a finite group, $x$ might not be 0!

Look at our $d_2$ matrix for $\mathbb{RP}^2$. It maps the face $f$ to $2e$. 
- The edge $e$ is a cycle (it has no boundary).
- Is $e$ a boundary? No, because $d_2(f) = 2e$. Only *twice* the edge is a boundary.

This creates **Torsion**. The cycle $e$ represents a hole, but if you wrap around it twice, the hole disappears! This is a $\mathbb{Z}_2$ torsion group.

To compute this algorithmically, `pysurgery` implements the **Smith Normal Form (SNF)**. The SNF algorithm applies invertible integer row and column operations (using the Extended Euclidean Algorithm) to diagonalize the boundary matrix into a form $S = U A V$. 
The diagonal entries $s_i$ (where $s_i$ divides $s_{i+1}$) are the **invariant factors**. 
- The number of zeros on the diagonal gives us the free rank (Betti number).
- The entries $s_i > 1$ give us the exact torsion coefficients (the twists)!

In [10]:
print("--- Computing Exact Integer Homology H_n(RP^2; Z) ---")
for n in [0, 1, 2]:
    # pysurgery calls the heavily optimized SNF engine here.
    rank, torsion = chain.homology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H_{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H_0 = Z^1 : The free rank is 1. The space has exactly 1 connected component.
# H_1 = Z_2 : The free rank is 0, but there is a Z_2 torsion factor. This perfectly captures the non-orientable twist!
# H_2 = 0   : There is no 2D cavity. The space is not orientable, so it has no fundamental class in H_2.

--- Computing Exact Integer Homology H_n(RP^2; Z) ---
H_0(RP^2) = Z^1
H_1(RP^2) = Z_2
H_2(RP^2) = 0


## 5. Cohomology and the Universal Coefficient Theorem (UCT)

For advanced operations like Surgery Theory and Intersection Forms (Tutorial 2), Homology is not enough. We need **Cohomology** $H^n(X)$.

Cohomology is the dual vector space. Instead of geometric chains, it consists of cochains: linear functions that evaluate chains and return an integer. The boundary operator is transposed $d_{n+1}^T$, becoming the "coboundary" operator $\delta_n$.

Why do we need it? Because Cohomology forms a **Ring**. You can multiply two cohomology classes together using the Cup Product ($\smile$). You cannot multiply homology classes.

### The Universal Coefficient Theorem
`pysurgery` computes Cohomology elegantly using the **Universal Coefficient Theorem (UCT)**. The UCT states that the cohomology group over $\mathbb{Z}$ is determined entirely by the homology groups:

$$H^n(X; \mathbb{Z}) \cong \operatorname{Free}(H_n(X)) \oplus \operatorname{Torsion}(H_{n-1}(X))$$

Look closely at the formula. The free parts (Betti numbers) stay in the same dimension. But the **Torsion shifts UP a dimension!** The twist that existed in 1D homology will mathematically teleport into 2D cohomology.

Let's verify this mathematically.

In [11]:
print("\n--- Computing Cohomology H^n(RP^2; Z) ---")
for n in [0, 1, 2]:
    rank, torsion = chain.cohomology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H^{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H^0 = Z^1 : Inherits the free Z from H_0.
# H^1 = 0   : The free part of H_1 is 0. H_0 has no torsion. Therefore, H^1 is totally empty.
# H^2 = Z_2 : The free part of H_2 is 0. But it inherits the Z_2 torsion from H_1! The torsion shifted up.


--- Computing Cohomology H^n(RP^2; Z) ---
H^0(RP^2) = Z^1
H^1(RP^2) = 0
H^2(RP^2) = Z_2


## Conclusion

You have successfully learned the deep mathematical foundations of `pysurgery`. We traversed from the idea of discrete cells to rigorous algebraic boundary matrices, and used the algorithmic power of the Smith Normal Form to extract exact Homology and Cohomology groups, perfectly handling complex topological torsion.

In the next tutorial (`02_intersection_forms.ipynb`), we will step into 4-dimensional topology and discover why counting holes isn't enough, introducing the matrix "DNA" that classifies 4-manifolds.

## --- Integration from Original: 01_basic_homology_and_cohomology.ipynb ---

This section contains the deep-dive visualization, plotting functions, and didactic examples merged from the original `01_basic_homology_and_cohomology.ipynb`.

## 1. From Discrete Data to Simplicial Complexes

In the real world, you rarely are handed a perfect mathematical equation of a manifold. Instead, you are handed **discrete data**: a point cloud from a 3D scanner, a dataset of high-dimensional vectors, or a network of nodes.

How do we do topology on discrete points? A point cloud has no topology; it's just disconnected dots (zero-dimensional topology). To find the shape of the data, we must connect the dots. We do this by building a **Simplicial Complex**.

- **Vietoris-Rips Complex**: Imagine growing a ball of radius $\epsilon$ around every point in your dataset. If two balls overlap, you draw an edge (a 1-simplex) between the points. If three balls mutually overlap, you fill in the triangle between them (a 2-simplex). If $k$ balls overlap, you fill in a $(k-1)$-dimensional solid simplex. This transforms discrete points into a continuous geometric shape!
- **Alpha Complex**: Similar to Rips, but geometrically constrained by the Delaunay triangulation. It is much more memory efficient for 2D and 3D data because it prevents the combinatorial explosion of high-dimensional simplices.

While `pysurgery` interfaces with libraries like `gudhi` to build these complexes from data (as we will see in Tutorial 4), at its core, `pysurgery` operates on the purely algebraic skeleton of these shapes: **The Chain Complex**.

## 2. The Chain Complex and Boundary Matrices

Once we have a Simplicial Complex (or a more general CW Complex), we organize its building blocks by dimension:
- $C_0$: The set of all vertices (0-cells).
- $C_1$: The set of all edges (1-cells).
- $C_2$: The set of all triangles/faces (2-cells).
- $C_n$: The set of all $n$-dimensional solid bodies.

To understand the topology, we must know exactly how an $n$-dimensional cell is glued to the $(n-1)$-dimensional cells. We represent this mathematically using the **Boundary Operator**, denoted as $\partial_n$ or $d_n$.

$$ d_n : C_n \to C_{n-1} $$

### How do we build the Boundary Matrix?
Suppose you have a triangle (a 2-simplex) made of vertices $[v_0, v_1, v_2]$. Its boundary consists of three edges: $[v_0, v_1]$, $[v_1, v_2]$, and $[v_0, v_2]$.

**Orientation is the most critical concept here.** If we just say the boundary is edge 1 + edge 2 + edge 3, we lose track of "direction." If you walk around the perimeter of the triangle, you traverse $[v_0, v_1]$, then $[v_1, v_2]$, but you traverse $[v_0, v_2]$ *backwards*. Therefore, the algebraic boundary is:
$$ \partial([v_0, v_1, v_2]) = [v_1, v_2] - [v_0, v_2] + [v_0, v_1] $$

Notice the alternating signs! $+1, -1, +1$. This alternating sum guarantees a fundamental theorem of topology: **The boundary of a boundary is zero** ($d_{n-1} \circ d_n = 0$).

In code, $d_n$ is a sparse matrix. The rows represent the $(n-1)$-cells, and the columns represent the $n$-cells. The entries are $0, 1,$ or $-1$ depending on how they are glued.

In [12]:
import numpy as np
from scipy.sparse import csr_matrix
from pysurgery.core.complexes import CWComplex

print("Libraries loaded. Ready to build a manifold.")

Libraries loaded. Ready to build a manifold.


## 3. Manually Constructing The Real Projective Plane ($\mathbb{RP}^2$)

To see how `pysurgery` handles complex algebra without needing a million data points, we will manually define a famous topological space: **The Real Projective Plane ($\mathbb{RP}^2$)**.

$\mathbb{RP}^2$ is a non-orientable surface. You cannot embed it in 3D space without it intersecting itself (like a closed Möbius strip).

We can build it using exactly 3 cells:
- One 0-cell: A vertex $v$.
- One 1-cell: An edge $e$.
- One 2-cell: A solid disk $f$.

**The Gluing Instructions (The Math):**
1. We take the edge $e$ and glue both of its ends to the single vertex $v$. This makes $e$ a loop. Thus, its boundary is $v - v = 0$. So, $d_1 = [0]$.
2. We take the solid disk $f$. Its boundary is a circle. We wrap this boundary circle around the loop $e$ **twice**. Thus, the boundary of $f$ is $2e$. So, $d_2 = [2]$.

Let's write this in code.

In [13]:
# Define the number of cells in each dimension
cells = {0: 1, 1: 1, 2: 1}

# Construct the Boundary Matrices using Scipy Sparse Matrices.
# We use sparse matrices because in real TDA data, these matrices can have millions of rows/cols,
# but are 99.9% zeros. Dense matrices would instantly crash your RAM.
d1 = csr_matrix((1, 1), dtype=np.int64)  # Matrix of zeros, shape (1,1)
d2 = csr_matrix([[2]], dtype=np.int64)  # Matrix with a single '2', shape (1,1)

attaching_maps = {1: d1, 2: d2}

# Instantiate the CWComplex
rp2 = CWComplex(cells=cells, attaching_maps=attaching_maps)

# Extract the purely algebraic Chain Complex
chain = rp2.cellular_chain_complex()
print("Chain Complex for RP^2 created successfully.")

Chain Complex for RP^2 created successfully.


## 4. The Algebra of Homology: Cycles vs Boundaries

Homology $H_n(X)$ is the mathematical formalization of "counting holes".

- **Cycles ($Z_n$)**: A collection of $n$-cells that have no boundary. For example, a closed loop of edges has no start or end point. Mathematically, these are the vectors in the **kernel** (nullspace) of $d_n$. ($Z_n = \ker(d_n)$).
- **Boundaries ($B_n$)**: A collection of $n$-cells that form the boundary of an $(n+1)$-cell. For example, the outer shell of a solid sphere is a cycle, but it is also a boundary (it's filled in). These are the vectors in the **image** (column space) of $d_{n+1}$. ($B_n = \operatorname{im}(d_{n+1})$).

A "hole" exists if there is a cycle that is **not** a boundary. You have a loop, but there is no solid disk filling it in.

Therefore, the Homology group is the quotient space:
$$ H_n(X) = Z_n / B_n = \ker(d_n) / \operatorname{im}(d_{n+1}) $$

### Why is computing this so hard? (The Smith Normal Form)
If we were doing linear algebra over the Real numbers $\mathbb{R}$, finding dimensions of kernels and images is easy (just use Gaussian Elimination / RREF). 

But we are doing topology. We must work over the **Integers $\mathbb{Z}$**. The integers form a Principal Ideal Domain (PID), not a Field. You cannot divide by 2. If you have $2x = 0$, over $\mathbb{R}$, $x=0$. Over a finite group, $x$ might not be 0!

Look at our $d_2$ matrix for $\mathbb{RP}^2$. It maps the face $f$ to $2e$. 
- The edge $e$ is a cycle (it has no boundary).
- Is $e$ a boundary? No, because $d_2(f) = 2e$. Only *twice* the edge is a boundary.

This creates **Torsion**. The cycle $e$ represents a hole, but if you wrap around it twice, the hole disappears! This is a $\mathbb{Z}_2$ torsion group.

To compute this algorithmically, `pysurgery` implements the **Smith Normal Form (SNF)**. The SNF algorithm applies invertible integer row and column operations (using the Extended Euclidean Algorithm) to diagonalize the boundary matrix into a form $S = U A V$. 
The diagonal entries $s_i$ (where $s_i$ divides $s_{i+1}$) are the **invariant factors**. 
- The number of zeros on the diagonal gives us the free rank (Betti number).
- The entries $s_i > 1$ give us the exact torsion coefficients (the twists)!

In [14]:
print("--- Computing Exact Integer Homology H_n(RP^2; Z) ---")
for n in [0, 1, 2]:
    # pysurgery calls the heavily optimized SNF engine here.
    rank, torsion = chain.homology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H_{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H_0 = Z^1 : The free rank is 1. The space has exactly 1 connected component.
# H_1 = Z_2 : The free rank is 0, but there is a Z_2 torsion factor. This perfectly captures the non-orientable twist!
# H_2 = 0   : There is no 2D cavity. The space is not orientable, so it has no fundamental class in H_2.

--- Computing Exact Integer Homology H_n(RP^2; Z) ---
H_0(RP^2) = Z^1
H_1(RP^2) = Z_2
H_2(RP^2) = 0


## 5. Cohomology and the Universal Coefficient Theorem (UCT)

For advanced operations like Surgery Theory and Intersection Forms (Tutorial 2), Homology is not enough. We need **Cohomology** $H^n(X)$.

Cohomology is the dual vector space. Instead of geometric chains, it consists of cochains: linear functions that evaluate chains and return an integer. The boundary operator is transposed $d_{n+1}^T$, becoming the "coboundary" operator $\delta_n$.

Why do we need it? Because Cohomology forms a **Ring**. You can multiply two cohomology classes together using the Cup Product ($\smile$). You cannot multiply homology classes.

### The Universal Coefficient Theorem
`pysurgery` computes Cohomology elegantly using the **Universal Coefficient Theorem (UCT)**. The UCT states that the cohomology group over $\mathbb{Z}$ is determined entirely by the homology groups:

$$H^n(X; \mathbb{Z}) \cong \operatorname{Free}(H_n(X)) \oplus \operatorname{Torsion}(H_{n-1}(X))$$

Look closely at the formula. The free parts (Betti numbers) stay in the same dimension. But the **Torsion shifts UP a dimension!** The twist that existed in 1D homology will mathematically teleport into 2D cohomology.

Let's verify this mathematically.

In [15]:
print("\n--- Computing Cohomology H^n(RP^2; Z) ---")
for n in [0, 1, 2]:
    rank, torsion = chain.cohomology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H^{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H^0 = Z^1 : Inherits the free Z from H_0.
# H^1 = 0   : The free part of H_1 is 0. H_0 has no torsion. Therefore, H^1 is totally empty.
# H^2 = Z_2 : The free part of H_2 is 0. But it inherits the Z_2 torsion from H_1! The torsion shifted up.


--- Computing Cohomology H^n(RP^2; Z) ---
H^0(RP^2) = Z^1
H^1(RP^2) = 0
H^2(RP^2) = Z_2


## Conclusion

You have successfully learned the deep mathematical foundations of `pysurgery`. We traversed from the idea of discrete cells to rigorous algebraic boundary matrices, and used the algorithmic power of the Smith Normal Form to extract exact Homology and Cohomology groups, perfectly handling complex topological torsion.

In the next tutorial (`02_intersection_forms.ipynb`), we will step into 4-dimensional topology and discover why counting holes isn't enough, introducing the matrix "DNA" that classifies 4-manifolds.

## --- Integration from Original: 01_basic_homology_and_cohomology.ipynb ---

This section contains the deep-dive visualization, plotting functions, and didactic examples merged from the original .

## 1. From Discrete Data to Simplicial Complexes

In the real world, you rarely are handed a perfect mathematical equation of a manifold. Instead, you are handed **discrete data**: a point cloud from a 3D scanner, a dataset of high-dimensional vectors, or a network of nodes.

How do we do topology on discrete points? A point cloud has no topology; it's just disconnected dots (zero-dimensional topology). To find the shape of the data, we must connect the dots. We do this by building a **Simplicial Complex**.

- **Vietoris-Rips Complex**: Imagine growing a ball of radius $\epsilon$ around every point in your dataset. If two balls overlap, you draw an edge (a 1-simplex) between the points. If three balls mutually overlap, you fill in the triangle between them (a 2-simplex). If $k$ balls overlap, you fill in a $(k-1)$-dimensional solid simplex. This transforms discrete points into a continuous geometric shape!
- **Alpha Complex**: Similar to Rips, but geometrically constrained by the Delaunay triangulation. It is much more memory efficient for 2D and 3D data because it prevents the combinatorial explosion of high-dimensional simplices.

While `pysurgery` interfaces with libraries like `gudhi` to build these complexes from data (as we will see in Tutorial 4), at its core, `pysurgery` operates on the purely algebraic skeleton of these shapes: **The Chain Complex**.

## 2. The Chain Complex and Boundary Matrices

Once we have a Simplicial Complex (or a more general CW Complex), we organize its building blocks by dimension:
- $C_0$: The set of all vertices (0-cells).
- $C_1$: The set of all edges (1-cells).
- $C_2$: The set of all triangles/faces (2-cells).
- $C_n$: The set of all $n$-dimensional solid bodies.

To understand the topology, we must know exactly how an $n$-dimensional cell is glued to the $(n-1)$-dimensional cells. We represent this mathematically using the **Boundary Operator**, denoted as $\partial_n$ or $d_n$.

$$ d_n : C_n \to C_{n-1} $$

### How do we build the Boundary Matrix?
Suppose you have a triangle (a 2-simplex) made of vertices $[v_0, v_1, v_2]$. Its boundary consists of three edges: $[v_0, v_1]$, $[v_1, v_2]$, and $[v_0, v_2]$.

**Orientation is the most critical concept here.** If we just say the boundary is edge 1 + edge 2 + edge 3, we lose track of "direction." If you walk around the perimeter of the triangle, you traverse $[v_0, v_1]$, then $[v_1, v_2]$, but you traverse $[v_0, v_2]$ *backwards*. Therefore, the algebraic boundary is:
$$ \partial([v_0, v_1, v_2]) = [v_1, v_2] - [v_0, v_2] + [v_0, v_1] $$

Notice the alternating signs! $+1, -1, +1$. This alternating sum guarantees a fundamental theorem of topology: **The boundary of a boundary is zero** ($d_{n-1} \circ d_n = 0$).

In code, $d_n$ is a sparse matrix. The rows represent the $(n-1)$-cells, and the columns represent the $n$-cells. The entries are $0, 1,$ or $-1$ depending on how they are glued.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from pysurgery.core.complexes import CWComplex

print("Libraries loaded. Ready to build a manifold.")

## 3. Manually Constructing The Real Projective Plane ($\mathbb{RP}^2$)

To see how `pysurgery` handles complex algebra without needing a million data points, we will manually define a famous topological space: **The Real Projective Plane ($\mathbb{RP}^2$)**.

$\mathbb{RP}^2$ is a non-orientable surface. You cannot embed it in 3D space without it intersecting itself (like a closed Möbius strip).

We can build it using exactly 3 cells:
- One 0-cell: A vertex $v$.
- One 1-cell: An edge $e$.
- One 2-cell: A solid disk $f$.

**The Gluing Instructions (The Math):**
1. We take the edge $e$ and glue both of its ends to the single vertex $v$. This makes $e$ a loop. Thus, its boundary is $v - v = 0$. So, $d_1 = [0]$.
2. We take the solid disk $f$. Its boundary is a circle. We wrap this boundary circle around the loop $e$ **twice**. Thus, the boundary of $f$ is $2e$. So, $d_2 = [2]$.

Let's write this in code.

In [ ]:
# Define the number of cells in each dimension
cells = {0: 1, 1: 1, 2: 1}

# Construct the Boundary Matrices using Scipy Sparse Matrices.
# We use sparse matrices because in real TDA data, these matrices can have millions of rows/cols,
# but are 99.9% zeros. Dense matrices would instantly crash your RAM.
d1 = csr_matrix((1, 1), dtype=np.int64)  # Matrix of zeros, shape (1,1)
d2 = csr_matrix([[2]], dtype=np.int64)  # Matrix with a single '2', shape (1,1)

attaching_maps = {1: d1, 2: d2}

# Instantiate the CWComplex
rp2 = CWComplex(cells=cells, attaching_maps=attaching_maps)

# Extract the purely algebraic Chain Complex
chain = rp2.cellular_chain_complex()
print("Chain Complex for RP^2 created successfully.")

## 4. The Algebra of Homology: Cycles vs Boundaries

Homology $H_n(X)$ is the mathematical formalization of "counting holes".

- **Cycles ($Z_n$)**: A collection of $n$-cells that have no boundary. For example, a closed loop of edges has no start or end point. Mathematically, these are the vectors in the **kernel** (nullspace) of $d_n$. ($Z_n = \ker(d_n)$).
- **Boundaries ($B_n$)**: A collection of $n$-cells that form the boundary of an $(n+1)$-cell. For example, the outer shell of a solid sphere is a cycle, but it is also a boundary (it's filled in). These are the vectors in the **image** (column space) of $d_{n+1}$. ($B_n = \operatorname{im}(d_{n+1})$).

A "hole" exists if there is a cycle that is **not** a boundary. You have a loop, but there is no solid disk filling it in.

Therefore, the Homology group is the quotient space:
$$ H_n(X) = Z_n / B_n = \ker(d_n) / \operatorname{im}(d_{n+1}) $$

### Why is computing this so hard? (The Smith Normal Form)
If we were doing linear algebra over the Real numbers $\mathbb{R}$, finding dimensions of kernels and images is easy (just use Gaussian Elimination / RREF).

But we are doing topology. We must work over the **Integers $\mathbb{Z}$**. The integers form a Principal Ideal Domain (PID), not a Field. You cannot divide by 2. If you have $2x = 0$, over $\mathbb{R}$, $x=0$. Over a finite group, $x$ might not be 0!

Look at our $d_2$ matrix for $\mathbb{RP}^2$. It maps the face $f$ to $2e$.
- The edge $e$ is a cycle (it has no boundary).
- Is $e$ a boundary? No, because $d_2(f) = 2e$. Only *twice* the edge is a boundary.

This creates **Torsion**. The cycle $e$ represents a hole, but if you wrap around it twice, the hole disappears! This is a $\mathbb{Z}_2$ torsion group.

To compute this algorithmically, `pysurgery` implements the **Smith Normal Form (SNF)**. The SNF algorithm applies invertible integer row and column operations (using the Extended Euclidean Algorithm) to diagonalize the boundary matrix into a form $S = U A V$.
The diagonal entries $s_i$ (where $s_i$ divides $s_{i+1}$) are the **invariant factors**.
- The number of zeros on the diagonal gives us the free rank (Betti number).
- The entries $s_i > 1$ give us the exact torsion coefficients (the twists)!

In [ ]:
print("--- Computing Exact Integer Homology H_n(RP^2; Z) ---")
for n in [0, 1, 2]:
    # pysurgery calls the heavily optimized SNF engine here.
    rank, torsion = chain.homology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H_{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H_0 = Z^1 : The free rank is 1. The space has exactly 1 connected component.
# H_1 = Z_2 : The free rank is 0, but there is a Z_2 torsion factor. This perfectly captures the non-orientable twist!
# H_2 = 0   : There is no 2D cavity. The space is not orientable, so it has no fundamental class in H_2.

## 5. Cohomology and the Universal Coefficient Theorem (UCT)

For advanced operations like Surgery Theory and Intersection Forms (Tutorial 2), Homology is not enough. We need **Cohomology** $H^n(X)$.

Cohomology is the dual vector space. Instead of geometric chains, it consists of cochains: linear functions that evaluate chains and return an integer. The boundary operator is transposed $d_{n+1}^T$, becoming the "coboundary" operator $\delta_n$.

Why do we need it? Because Cohomology forms a **Ring**. You can multiply two cohomology classes together using the Cup Product ($\smile$). You cannot multiply homology classes.

### The Universal Coefficient Theorem
`pysurgery` computes Cohomology elegantly using the **Universal Coefficient Theorem (UCT)**. The UCT states that the cohomology group over $\mathbb{Z}$ is determined entirely by the homology groups:

$$H^n(X; \mathbb{Z}) \cong \operatorname{Free}(H_n(X)) \oplus \operatorname{Torsion}(H_{n-1}(X))$$

Look closely at the formula. The free parts (Betti numbers) stay in the same dimension. But the **Torsion shifts UP a dimension!** The twist that existed in 1D homology will mathematically teleport into 2D cohomology.

Let's verify this mathematically.

In [ ]:
print("\n--- Computing Cohomology H^n(RP^2; Z) ---")
for n in [0, 1, 2]:
    rank, torsion = chain.cohomology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H^{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H^0 = Z^1 : Inherits the free Z from H_0.
# H^1 = 0   : The free part of H_1 is 0. H_0 has no torsion. Therefore, H^1 is totally empty.
# H^2 = Z_2 : The free part of H_2 is 0. But it inherits the Z_2 torsion from H_1! The torsion shifted up.

## Conclusion

You have successfully learned the deep mathematical foundations of `pysurgery`. We traversed from the idea of discrete cells to rigorous algebraic boundary matrices, and used the algorithmic power of the Smith Normal Form to extract exact Homology and Cohomology groups, perfectly handling complex topological torsion.

In the next tutorial (`02_intersection_forms.ipynb`), we will step into 4-dimensional topology and discover why counting holes isn't enough, introducing the matrix "DNA" that classifies 4-manifolds.


## --- Original Deep Dive: Visualizations and Didactic Content ---

Appended from original `01_basic_homology_and_cohomology.ipynb`

## 1. From Discrete Data to Simplicial Complexes

In the real world, you rarely are handed a perfect mathematical equation of a manifold. Instead, you are handed **discrete data**: a point cloud from a 3D scanner, a dataset of high-dimensional vectors, or a network of nodes.

How do we do topology on discrete points? A point cloud has no topology; it's just disconnected dots (zero-dimensional topology). To find the shape of the data, we must connect the dots. We do this by building a **Simplicial Complex**.

- **Vietoris-Rips Complex**: Imagine growing a ball of radius $\epsilon$ around every point in your dataset. If two balls overlap, you draw an edge (a 1-simplex) between the points. If three balls mutually overlap, you fill in the triangle between them (a 2-simplex). If $k$ balls overlap, you fill in a $(k-1)$-dimensional solid simplex. This transforms discrete points into a continuous geometric shape!
- **Alpha Complex**: Similar to Rips, but geometrically constrained by the Delaunay triangulation. It is much more memory efficient for 2D and 3D data because it prevents the combinatorial explosion of high-dimensional simplices.

While `pysurgery` interfaces with libraries like `gudhi` to build these complexes from data (as we will see in Tutorial 4), at its core, `pysurgery` operates on the purely algebraic skeleton of these shapes: **The Chain Complex**.

## 2. The Chain Complex and Boundary Matrices

Once we have a Simplicial Complex (or a more general CW Complex), we organize its building blocks by dimension:
- $C_0$: The set of all vertices (0-cells).
- $C_1$: The set of all edges (1-cells).
- $C_2$: The set of all triangles/faces (2-cells).
- $C_n$: The set of all $n$-dimensional solid bodies.

To understand the topology, we must know exactly how an $n$-dimensional cell is glued to the $(n-1)$-dimensional cells. We represent this mathematically using the **Boundary Operator**, denoted as $\partial_n$ or $d_n$.

$$ d_n : C_n \to C_{n-1} $$

### How do we build the Boundary Matrix?
Suppose you have a triangle (a 2-simplex) made of vertices $[v_0, v_1, v_2]$. Its boundary consists of three edges: $[v_0, v_1]$, $[v_1, v_2]$, and $[v_0, v_2]$.

**Orientation is the most critical concept here.** If we just say the boundary is edge 1 + edge 2 + edge 3, we lose track of "direction." If you walk around the perimeter of the triangle, you traverse $[v_0, v_1]$, then $[v_1, v_2]$, but you traverse $[v_0, v_2]$ *backwards*. Therefore, the algebraic boundary is:
$$ \partial([v_0, v_1, v_2]) = [v_1, v_2] - [v_0, v_2] + [v_0, v_1] $$

Notice the alternating signs! $+1, -1, +1$. This alternating sum guarantees a fundamental theorem of topology: **The boundary of a boundary is zero** ($d_{n-1} \circ d_n = 0$).

In code, $d_n$ is a sparse matrix. The rows represent the $(n-1)$-cells, and the columns represent the $n$-cells. The entries are $0, 1,$ or $-1$ depending on how they are glued.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from pysurgery.core.complexes import CWComplex

print("Libraries loaded. Ready to build a manifold.")

## 3. Manually Constructing The Real Projective Plane ($\mathbb{RP}^2$)

To see how `pysurgery` handles complex algebra without needing a million data points, we will manually define a famous topological space: **The Real Projective Plane ($\mathbb{RP}^2$)**.

$\mathbb{RP}^2$ is a non-orientable surface. You cannot embed it in 3D space without it intersecting itself (like a closed Möbius strip).

We can build it using exactly 3 cells:
- One 0-cell: A vertex $v$.
- One 1-cell: An edge $e$.
- One 2-cell: A solid disk $f$.

**The Gluing Instructions (The Math):**
1. We take the edge $e$ and glue both of its ends to the single vertex $v$. This makes $e$ a loop. Thus, its boundary is $v - v = 0$. So, $d_1 = [0]$.
2. We take the solid disk $f$. Its boundary is a circle. We wrap this boundary circle around the loop $e$ **twice**. Thus, the boundary of $f$ is $2e$. So, $d_2 = [2]$.

Let's write this in code.

In [ ]:
# Define the number of cells in each dimension
cells = {0: 1, 1: 1, 2: 1}

# Construct the Boundary Matrices using Scipy Sparse Matrices.
# We use sparse matrices because in real TDA data, these matrices can have millions of rows/cols,
# but are 99.9% zeros. Dense matrices would instantly crash your RAM.
d1 = csr_matrix((1, 1), dtype=np.int64)  # Matrix of zeros, shape (1,1)
d2 = csr_matrix([[2]], dtype=np.int64)  # Matrix with a single '2', shape (1,1)

attaching_maps = {1: d1, 2: d2}

# Instantiate the CWComplex
rp2 = CWComplex(cells=cells, attaching_maps=attaching_maps)

# Extract the purely algebraic Chain Complex
chain = rp2.cellular_chain_complex()
print("Chain Complex for RP^2 created successfully.")

## 4. The Algebra of Homology: Cycles vs Boundaries

Homology $H_n(X)$ is the mathematical formalization of "counting holes".

- **Cycles ($Z_n$)**: A collection of $n$-cells that have no boundary. For example, a closed loop of edges has no start or end point. Mathematically, these are the vectors in the **kernel** (nullspace) of $d_n$. ($Z_n = \ker(d_n)$).
- **Boundaries ($B_n$)**: A collection of $n$-cells that form the boundary of an $(n+1)$-cell. For example, the outer shell of a solid sphere is a cycle, but it is also a boundary (it's filled in). These are the vectors in the **image** (column space) of $d_{n+1}$. ($B_n = \operatorname{im}(d_{n+1})$).

A "hole" exists if there is a cycle that is **not** a boundary. You have a loop, but there is no solid disk filling it in.

Therefore, the Homology group is the quotient space:
$$ H_n(X) = Z_n / B_n = \ker(d_n) / \operatorname{im}(d_{n+1}) $$

### Why is computing this so hard? (The Smith Normal Form)
If we were doing linear algebra over the Real numbers $\mathbb{R}$, finding dimensions of kernels and images is easy (just use Gaussian Elimination / RREF). 

But we are doing topology. We must work over the **Integers $\mathbb{Z}$**. The integers form a Principal Ideal Domain (PID), not a Field. You cannot divide by 2. If you have $2x = 0$, over $\mathbb{R}$, $x=0$. Over a finite group, $x$ might not be 0!

Look at our $d_2$ matrix for $\mathbb{RP}^2$. It maps the face $f$ to $2e$. 
- The edge $e$ is a cycle (it has no boundary).
- Is $e$ a boundary? No, because $d_2(f) = 2e$. Only *twice* the edge is a boundary.

This creates **Torsion**. The cycle $e$ represents a hole, but if you wrap around it twice, the hole disappears! This is a $\mathbb{Z}_2$ torsion group.

To compute this algorithmically, `pysurgery` implements the **Smith Normal Form (SNF)**. The SNF algorithm applies invertible integer row and column operations (using the Extended Euclidean Algorithm) to diagonalize the boundary matrix into a form $S = U A V$. 
The diagonal entries $s_i$ (where $s_i$ divides $s_{i+1}$) are the **invariant factors**. 
- The number of zeros on the diagonal gives us the free rank (Betti number).
- The entries $s_i > 1$ give us the exact torsion coefficients (the twists)!

In [ ]:
print("--- Computing Exact Integer Homology H_n(RP^2; Z) ---")
for n in [0, 1, 2]:
    # pysurgery calls the heavily optimized SNF engine here.
    rank, torsion = chain.homology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H_{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H_0 = Z^1 : The free rank is 1. The space has exactly 1 connected component.
# H_1 = Z_2 : The free rank is 0, but there is a Z_2 torsion factor. This perfectly captures the non-orientable twist!
# H_2 = 0   : There is no 2D cavity. The space is not orientable, so it has no fundamental class in H_2.

## 5. Cohomology and the Universal Coefficient Theorem (UCT)

For advanced operations like Surgery Theory and Intersection Forms (Tutorial 2), Homology is not enough. We need **Cohomology** $H^n(X)$.

Cohomology is the dual vector space. Instead of geometric chains, it consists of cochains: linear functions that evaluate chains and return an integer. The boundary operator is transposed $d_{n+1}^T$, becoming the "coboundary" operator $\delta_n$.

Why do we need it? Because Cohomology forms a **Ring**. You can multiply two cohomology classes together using the Cup Product ($\smile$). You cannot multiply homology classes.

### The Universal Coefficient Theorem
`pysurgery` computes Cohomology elegantly using the **Universal Coefficient Theorem (UCT)**. The UCT states that the cohomology group over $\mathbb{Z}$ is determined entirely by the homology groups:

$$H^n(X; \mathbb{Z}) \cong \operatorname{Free}(H_n(X)) \oplus \operatorname{Torsion}(H_{n-1}(X))$$

Look closely at the formula. The free parts (Betti numbers) stay in the same dimension. But the **Torsion shifts UP a dimension!** The twist that existed in 1D homology will mathematically teleport into 2D cohomology.

Let's verify this mathematically.

In [ ]:
print("\n--- Computing Cohomology H^n(RP^2; Z) ---")
for n in [0, 1, 2]:
    rank, torsion = chain.cohomology(n)

    components = []
    if rank > 0:
        components.append(f"Z^{rank}")
    if torsion:
        components.extend([f"Z_{t}" for t in torsion])

    group_str = " + ".join(components) if components else "0"
    print(f"H^{n}(RP^2) = {group_str}")

# EXPECTED OUTPUT EXPLANATION:
# H^0 = Z^1 : Inherits the free Z from H_0.
# H^1 = 0   : The free part of H_1 is 0. H_0 has no torsion. Therefore, H^1 is totally empty.
# H^2 = Z_2 : The free part of H_2 is 0. But it inherits the Z_2 torsion from H_1! The torsion shifted up.

## Conclusion

You have successfully learned the deep mathematical foundations of `pysurgery`. We traversed from the idea of discrete cells to rigorous algebraic boundary matrices, and used the algorithmic power of the Smith Normal Form to extract exact Homology and Cohomology groups, perfectly handling complex topological torsion.

In the next tutorial (`02_intersection_forms.ipynb`), we will step into 4-dimensional topology and discover why counting holes isn't enough, introducing the matrix "DNA" that classifies 4-manifolds.

## Formal Grounding Checkpoint

- **Chain complex:** $C_*=(C_n,\partial_n)$ with $\partial_n \circ \partial_{n+1}=0$.
- **Homology:** $H_n(C_*)=\ker(\partial_n)/\operatorname{im}(\partial_{n+1})$.
- **Intersection form:** $Q_M:H_2(M;\mathbb{Z})\times H_2(M;\mathbb{Z})\to\mathbb{Z}$.
- **Decision statuses:** `success`, `impediment`, `inconclusive`, and `surgery_required` are tracked explicitly in `pysurgery`.

| Object | Formal definition | Pipeline role |
|---|---|---|
| Boundary map | $\partial_n:C_n\to C_{n-1}$ | Encodes orientation and incidence |
| Smith normal form | $UAV=\operatorname{diag}(d_1,\dots,d_r,0,\dots)$ with $d_i\mid d_{i+1}$ | Exact rank/torsion extraction |
| Structure set | $\mathcal{S}(M)$ | Tracks manifold classification choices in surgery |
